# 04 — UAI, AAVI and multiscale spatial-aggregation analysis

This notebook reproduces the final municipal analysis reported in the manuscript.

**Sequence**

1. load and validate the 31,664-building accessibility universe;
2. merge building-level estimates of residents aged 65 years and over;
3. repair only the four terrain-QC Tobler mean-time cases;
4. construct the eight-variable PCA matrix;
5. calculate KMO and Bartlett's test;
6. estimate PCA weights and the Urban Attractiveness Index (UAI);
7. test equal-weight, alternative-imputation and complete-case robustness;
8. calculate the Ageing Accessibility Vulnerability Index (AAVI);
9. evaluate Moran's I, LISA and KNN sensitivity;
10. compare buildings with 100 m, 250 m, 500 m and BGRI representations;
11. test priority retention at 5%, 10% and 20% thresholds;
12. export the manuscript tables, figures and final analytical dataset.

Population is **not** included in the PCA. The AAVI is a complementary territorial screening measure calculated only after the UAI.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.geometry import box
from scipy.stats import chi2, pearsonr, spearmanr
from scipy.sparse.csgraph import connected_components
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
SEED = 42
np.random.seed(SEED)
CRS_PROJECTED = 'EPSG:3763'
K_MAIN = 20
K_SENSITIVITY = [8, 12, 16, 20, 24, 32]
PERMUTATIONS = 999
PRIORITY_Q = 0.1
PRIORITY_THRESHOLDS = [0.05, 0.1, 0.2]
STRICT_MANUSCRIPT_VALIDATION = True

def resolve_repo_root():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
        return cwd
    for parent in cwd.parents:
        if (parent / 'notebooks').exists() and (parent / 'data').exists():
            return parent
    raise RuntimeError('Repository root not found. Run the notebook from the repository root or from the notebooks directory.')
REPO_ROOT = resolve_repo_root()
RAW_DATA = REPO_ROOT / 'data' / 'raw'
INTERMEDIATE_DATA = REPO_ROOT / 'data' / 'intermediate'
PROCESSED_DATA = REPO_ROOT / 'data' / 'processed'
RESULTS_TABLES = REPO_ROOT / 'results' / 'tables'
RESULTS_FIGURES = REPO_ROOT / 'results' / 'figures'
RESULTS_SPATIAL = REPO_ROOT / 'results' / 'spatial'
for directory in [PROCESSED_DATA, RESULTS_TABLES, RESULTS_FIGURES, RESULTS_SPATIAL]:
    directory.mkdir(parents=True, exist_ok=True)
ACCESS_FILE = PROCESSED_DATA / 'porto_building_accessibility_final.csv'
POP_FILE = INTERMEDIATE_DATA / 'population' / 'population_65plus_by_building.csv'
BGRI_FILE = RAW_DATA / 'BGRI2021_1312.gpkg'
BGRI_ID_COL = 'DTMNFRSEC21'
required_files = [ACCESS_FILE, POP_FILE, BGRI_FILE]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
print('Repository root:', REPO_ROOT)
print('Accessibility input:', ACCESS_FILE)
print('Population input:', POP_FILE)
print('BGRI input:', BGRI_FILE)

## 1. Load the analytical universe and run structural QC

The global accessibility file is converted to a GeoDataFrame, building IDs are harmonised, older-population estimates are joined, and the two upstream QC flags used by this notebook are reconstructed when necessary: zero-minimum snapping and invalid Tobler mean times.


In [ ]:
df = pd.read_csv(ACCESS_FILE, low_memory=False)
df['osm_id'] = df['osm_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip()
geom_col = 'geometry_wkt' if 'geometry_wkt' in df.columns else 'geometry'
if geom_col not in df.columns:
    raise KeyError('The accessibility CSV must contain geometry_wkt or geometry.')
geom = df[geom_col].apply(lambda x: wkt.loads(x) if pd.notna(x) and str(x).strip() else None)
valid_geom = geom.dropna()
if valid_geom.empty:
    raise ValueError('No valid geometries were found.')
sample_bounds = gpd.GeoSeries(valid_geom.iloc[:min(1000, len(valid_geom))]).total_bounds
looks_geographic = sample_bounds[0] >= -180 and sample_bounds[2] <= 180 and (sample_bounds[1] >= -90) and (sample_bounds[3] <= 90)
input_crs = 'EPSG:4326' if looks_geographic else CRS_PROJECTED
print('Inferred WKT CRS:', input_crs)
print('Sample bounds:', sample_bounds)
gdf = gpd.GeoDataFrame(df.copy(), geometry=geom, crs=input_crs)
gdf = gdf.dropna(subset=['geometry']).copy()
gdf = gdf[gdf.geometry.is_valid].copy().to_crs(CRS_PROJECTED)
gdf['geometry'] = gdf.geometry.buffer(0)
pop = pd.read_csv(POP_FILE, low_memory=False)
pop['osm_id'] = pop['osm_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip()
if 'pop_64_mais' not in pop.columns:
    raise KeyError('population_65plus_by_building.csv must contain pop_64_mais.')
pop_small = pop[['osm_id', 'pop_64_mais']].copy()
pop_small['pop_64_mais'] = pd.to_numeric(pop_small['pop_64_mais'], errors='coerce').fillna(0)
if pop_small['osm_id'].duplicated().any():
    pop_small = pop_small.groupby('osm_id', as_index=False)['pop_64_mais'].sum()
gdf = gdf.merge(pop_small, on='osm_id', how='left')
gdf['pop_64_mais'] = gdf['pop_64_mais'].fillna(0)
SERVICE_COLS = ['Supermercados', 'Bancos', 'Farmacias', 'CTT', 'Parques e jardins', 'Centro Saude', 'Hospitais']
for c in SERVICE_COLS + ['numero_servicos_proximos', 'distancia_media_servicos']:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors='coerce')
required_cols = SERVICE_COLS + ['numero_servicos_proximos', 'tempo_medio_seg__flat_0p7', 'tempo_medio_seg__tobler_0p7', 'ficheiro_origem']
missing_cols = [c for c in required_cols if c not in gdf.columns]
if missing_cols:
    raise KeyError('Missing required columns: ' + ', '.join(missing_cols))
for c in ['tempo_medio_seg__flat_0p7', 'tempo_medio_seg__tobler_0p7']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce')
gdf['no_destinations'] = gdf['numero_servicos_proximos'].fillna(0).eq(0)
if 'minimo_zero_snap' in gdf.columns:
    gdf['minimo_zero_snap'] = gdf['minimo_zero_snap'].astype(str).str.lower().isin(['true', '1'])
else:
    gdf['minimo_zero_snap'] = pd.to_numeric(gdf.get('distancia_minima_servico'), errors='coerce').eq(0)
if 'tobler_qc_excluded' in gdf.columns:
    gdf['tobler_qc_excluded'] = gdf['tobler_qc_excluded'].astype(str).str.lower().isin(['true', '1'])
else:
    gdf['tobler_qc_excluded'] = gdf['tempo_medio_seg__tobler_0p7'].gt(60 * 60)
print('\n================ ANALYTICAL UNIVERSE ================')
print('Buildings:', len(gdf))
print('Unique osm_id:', gdf['osm_id'].nunique())
print('Duplicate osm_id:', int(gdf['osm_id'].duplicated().sum()))
print('Buildings with no services <=800 m:', int(gdf['no_destinations'].sum()))
print('Zero-minimum snapping flags:', int(gdf['minimo_zero_snap'].sum()))
print('Invalid Tobler mean-time flags:', int(gdf['tobler_qc_excluded'].sum()))
print('Merged population aged 65+:', int(gdf['pop_64_mais'].sum()))
assert gdf['osm_id'].is_unique, 'Duplicate building IDs are not allowed.'
assert (gdf[SERVICE_COLS].fillna(0) >= 0).all().all(), 'Negative service counts detected.'

## 2. Build the slope-adjusted proximity variable and repair only the four invalid Tobler means

The main imputation is `flat mean time at 0.7 m/s × parish median Tobler/flat ratio`. A second version uses the global median ratio. These are used only for the four mean Tobler times invalidated by terrain QC; no building is removed from the analytical universe.


In [ ]:
flat = gdf['tempo_medio_seg__flat_0p7'].astype(float)
tobler = gdf['tempo_medio_seg__tobler_0p7'].astype(float)
valid_ratio = ~gdf['no_destinations'] & ~gdf['tobler_qc_excluded'] & flat.gt(0) & tobler.gt(0) & np.isfinite(flat) & np.isfinite(tobler)
gdf['ratio_tobler_flat_0p7'] = np.nan
gdf.loc[valid_ratio, 'ratio_tobler_flat_0p7'] = tobler.loc[valid_ratio] / flat.loc[valid_ratio]
global_ratio = float(gdf.loc[valid_ratio, 'ratio_tobler_flat_0p7'].median())
parish_ratio = gdf.loc[valid_ratio].groupby('ficheiro_origem')['ratio_tobler_flat_0p7'].median().to_dict()
print('Global median Tobler/flat ratio at 0.7 m/s:', round(global_ratio, 4))
print('\nParish medians:')
print(pd.Series(parish_ratio).sort_index().round(4))
gdf['tempo_medio_tobler07_qc_seg'] = tobler.copy()
gdf['tempo_medio_tobler07_globalimp_seg'] = tobler.copy()
mask_qc = gdf['tobler_qc_excluded'] & ~gdf['no_destinations']
for idx in gdf.index[mask_qc]:
    local_ratio = parish_ratio.get(gdf.at[idx, 'ficheiro_origem'], global_ratio)
    if not np.isfinite(local_ratio):
        local_ratio = global_ratio
    gdf.at[idx, 'tempo_medio_tobler07_qc_seg'] = gdf.at[idx, 'tempo_medio_seg__flat_0p7'] * local_ratio
    gdf.at[idx, 'tempo_medio_tobler07_globalimp_seg'] = gdf.at[idx, 'tempo_medio_seg__flat_0p7'] * global_ratio
print('\nQC-repaired cases:')
print(gdf.loc[mask_qc, ['osm_id', 'ficheiro_origem', 'tempo_medio_seg__flat_0p7', 'tempo_medio_seg__tobler_0p7', 'tempo_medio_tobler07_qc_seg', 'tempo_medio_tobler07_globalimp_seg']].to_string(index=False))
n_missing_valid = int((gdf['tempo_medio_tobler07_qc_seg'].isna() & ~gdf['no_destinations']).sum())
print('\nMissing repaired Tobler means among buildings with services:', n_missing_valid)
assert n_missing_valid == 0

## 3. Construct the eight-variable PCA matrix

Service variables are min-max normalised to [0,1]. Proximity is the reversed, min-max-normalised mean Tobler travel time at 0.7 m/s, so larger values always indicate more favourable accessibility. Buildings without any accessible service receive proximity = 0.


In [ ]:
PCA_MAP = {'supermarkets': 'Supermercados', 'banks': 'Bancos', 'pharmacies': 'Farmacias', 'post_offices': 'CTT', 'parks_gardens': 'Parques e jardins', 'health_centres': 'Centro Saude', 'hospitals': 'Hospitais'}
PCA_VARS = ['banks', 'supermarkets', 'pharmacies', 'post_offices', 'health_centres', 'parks_gardens', 'hospitals', 'proximity']
X_raw = pd.DataFrame(index=gdf.index)
for new, old in PCA_MAP.items():
    X_raw[new] = pd.to_numeric(gdf[old], errors='coerce').fillna(0).clip(lower=0)
X_norm = pd.DataFrame(index=gdf.index)
for c in X_raw.columns:
    mn, mx = (float(X_raw[c].min()), float(X_raw[c].max()))
    X_norm[c] = 0.0 if mx == mn else (X_raw[c] - mn) / (mx - mn)
valid_time = ~gdf['no_destinations'] & gdf['tempo_medio_tobler07_qc_seg'].notna()
t = gdf.loc[valid_time, 'tempo_medio_tobler07_qc_seg'].astype(float)
tmin, tmax = (float(t.min()), float(t.max()))
X_norm['proximity'] = 0.0
if tmax > tmin:
    X_norm.loc[valid_time, 'proximity'] = 1 - (t - tmin) / (tmax - tmin)
else:
    X_norm.loc[valid_time, 'proximity'] = 1.0
valid_time_alt = ~gdf['no_destinations'] & gdf['tempo_medio_tobler07_globalimp_seg'].notna()
t_alt = gdf.loc[valid_time_alt, 'tempo_medio_tobler07_globalimp_seg'].astype(float)
tmin_alt, tmax_alt = (float(t_alt.min()), float(t_alt.max()))
proximity_alt = pd.Series(0.0, index=gdf.index)
if tmax_alt > tmin_alt:
    proximity_alt.loc[valid_time_alt] = 1 - (t_alt - tmin_alt) / (tmax_alt - tmin_alt)
else:
    proximity_alt.loc[valid_time_alt] = 1.0
X_norm = X_norm[PCA_VARS].copy()
print('PCA matrix:', X_norm.shape)
print('Missing values:', int(X_norm.isna().sum().sum()))
print('Proximity range:', (float(X_norm['proximity'].min()), float(X_norm['proximity'].max())))
print('No-service buildings with proximity = 0:', int((X_norm.loc[gdf['no_destinations'], 'proximity'] == 0).sum()), '/', int(gdf['no_destinations'].sum()))
assert X_norm.shape[1] == 8
assert X_norm.isna().sum().sum() == 0
assert ((X_norm >= 0) & (X_norm <= 1)).all().all()
X_audit = X_norm.copy()
X_audit.insert(0, 'osm_id', gdf['osm_id'].values)
X_audit.to_csv(RESULTS_TABLES / '01_pca_normalized_matrix.csv', index=False)

## 4. PCA suitability: Bartlett's test and KMO

Bartlett's test checks whether the correlation matrix differs from an identity matrix. KMO is reported globally and for each variable.


In [ ]:
def bartlett_sphericity(X):
    X = np.asarray(X, dtype=float)
    n, p = X.shape
    R = np.corrcoef(X, rowvar=False)
    det_r = np.linalg.det(R)
    if det_r <= 0:
        raise ValueError('Correlation matrix determinant is not positive.')
    chi_square = -(n - 1 - (2 * p + 5) / 6) * np.log(det_r)
    df_b = p * (p - 1) / 2
    p_value = chi2.sf(chi_square, df_b)
    return (chi_square, int(df_b), p_value)

def kmo_test(X):
    X = np.asarray(X, dtype=float)
    R = np.corrcoef(X, rowvar=False)
    inv_r = np.linalg.pinv(R)
    partial = np.zeros_like(R)
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            if i != j:
                partial[i, j] = -inv_r[i, j] / np.sqrt(inv_r[i, i] * inv_r[j, j])
    r2 = R ** 2
    p2 = partial ** 2
    np.fill_diagonal(r2, 0)
    np.fill_diagonal(p2, 0)
    kmo_vars = r2.sum(axis=0) / (r2.sum(axis=0) + p2.sum(axis=0))
    kmo_total = r2.sum() / (r2.sum() + p2.sum())
    return (kmo_total, kmo_vars)
bartlett_chi2, bartlett_df, bartlett_p = bartlett_sphericity(X_norm)
kmo_total, kmo_vars = kmo_test(X_norm)
print('Bartlett chi2:', round(bartlett_chi2, 3), '| df:', bartlett_df, '| p:', bartlett_p)
print('KMO total:', round(float(kmo_total), 3))
print('\nKMO by variable:')
print(pd.Series(kmo_vars, index=X_norm.columns).sort_values().round(4))
pd.DataFrame({'variable': X_norm.columns, 'KMO': kmo_vars}).to_csv(RESULTS_TABLES / '02_kmo_by_variable.csv', index=False)

## 5. Estimate PCA weights and the Urban Attractiveness Index

The UAI is the weighted sum of the eight normalised variables. Weights are proportional to the absolute PC1 coefficients; because all coefficients within a component share the same eigenvalue scaling, this is equivalent for normalised weights to using conventional PC1 loadings. The final UAI is rescaled to [0,1].

The primary robustness check compares the PCA-weighted UAI with an equal-weight alternative using Pearson correlation, Spearman rank correlation, lower-decile retention and Jaccard similarity.


In [ ]:
pca_full = PCA(random_state=SEED).fit(X_norm)
explained = pca_full.explained_variance_ratio_
pc1_coefficients = pca_full.components_[0].copy()
if pc1_coefficients.sum() < 0:
    pc1_coefficients *= -1
weights = np.abs(pc1_coefficients)
weights = weights / weights.sum()
weights_df = pd.DataFrame({'variable': X_norm.columns, 'PC1_coefficient': pc1_coefficients, 'pca_weight': weights}).sort_values('pca_weight', ascending=False)
print('PC1 explained variance:', round(float(explained[0]), 6))
print('Cumulative PC1+PC2 variance:', round(float(explained[:2].sum()), 6))
print('\nPCA weights:')
display(weights_df.round(6))
gdf['uai_raw'] = np.dot(X_norm.values, weights)
gdf['uai'] = MinMaxScaler().fit_transform(gdf[['uai_raw']]).ravel()
equal_w = np.repeat(1 / len(weights), len(weights))
gdf['uai_equal_raw'] = np.dot(X_norm.values, equal_w)
gdf['uai_equal'] = MinMaxScaler().fit_transform(gdf[['uai_equal_raw']]).ravel()
pear_eq = pearsonr(gdf['uai'], gdf['uai_equal']).statistic
spear_eq = spearmanr(gdf['uai'], gdf['uai_equal']).statistic

def tie_aware_ids(df_in, value_col, id_col='osm_id', q=0.1):
    tmp = df_in[[id_col, value_col]].sort_values([value_col, id_col]).copy()
    n_target = int(np.ceil(len(tmp) * q))
    cutoff = float(tmp.iloc[n_target - 1][value_col])
    ids = set(tmp.loc[tmp[value_col] <= cutoff, id_col])
    return (ids, cutoff, n_target)
set_pca, cutoff_pca, target_pca = tie_aware_ids(gdf, 'uai', q=PRIORITY_Q)
set_eq, cutoff_eq, _ = tie_aware_ids(gdf, 'uai_equal', q=PRIORITY_Q)
priority_intersection_eq = set_pca & set_eq
priority_jaccard_equal = len(priority_intersection_eq) / len(set_pca | set_eq)
priority_ret_equal = len(priority_intersection_eq) / len(set_pca)
print('\nEqual-weight robustness:')
print('Pearson r:', round(pear_eq, 6))
print('Spearman rho:', round(spear_eq, 6))
print('Priority retention:', round(priority_ret_equal * 100, 3), '%')
print('Priority Jaccard:', round(priority_jaccard_equal, 6))
weights_df.to_csv(RESULTS_TABLES / '03_pca_weights.csv', index=False)
pd.DataFrame({'PC': np.arange(1, len(explained) + 1), 'variance_ratio': explained, 'cumulative_variance': np.cumsum(explained)}).to_csv(RESULTS_TABLES / '04_pca_explained_variance.csv', index=False)

## 6. Sensitivity to the treatment of the four invalid Tobler cases

Two complementary checks are used:

1. **Alternative imputation:** parish-median Tobler/flat ratio versus global-median ratio.
2. **Complete-case PCA:** re-estimate PCA, weights and UAI after removing the four QC cases entirely, then compare the common 31,660 buildings.

This prevents the analytical conclusion from depending on a single repair rule.


In [ ]:
X_alt = X_norm.copy()
X_alt['proximity'] = proximity_alt.values
pca_alt = PCA(random_state=SEED).fit(X_alt)
pc1_alt = pca_alt.components_[0].copy()
if pc1_alt.sum() < 0:
    pc1_alt *= -1
weights_alt = np.abs(pc1_alt)
weights_alt = weights_alt / weights_alt.sum()
uai_alt_raw = np.dot(X_alt.values, weights_alt)
uai_alt = MinMaxScaler().fit_transform(uai_alt_raw.reshape(-1, 1)).ravel()
pear_imp = pearsonr(gdf['uai'], uai_alt).statistic
spear_imp = spearmanr(gdf['uai'], uai_alt).statistic
alt_df = pd.DataFrame({'osm_id': gdf['osm_id'].values, 'uai_alt': uai_alt})
set_alt, _, _ = tie_aware_ids(alt_df, 'uai_alt', q=PRIORITY_Q)
inter_imp = set_pca & set_alt
ret_imp = len(inter_imp) / len(set_pca)
jac_imp = len(inter_imp) / len(set_pca | set_alt)
imputation_sensitivity = pd.DataFrame([{'comparison': 'parish-median ratio vs global-median ratio', 'Pearson_r': pear_imp, 'Spearman_rho': spear_imp, 'priority_retention_pct': ret_imp * 100, 'priority_Jaccard': jac_imp, 'PC1_variance_primary': explained[0], 'PC1_variance_alternative': pca_alt.explained_variance_ratio_[0]}])
display(imputation_sensitivity.round(8))
imputation_sensitivity.to_csv(RESULTS_TABLES / '05_imputation_sensitivity.csv', index=False)

In [ ]:
X_base = X_norm[PCA_VARS].copy().reset_index(drop=True)
ids = gdf['osm_id'].astype(str).str.replace('\\.0$', '', regex=True).str.strip().reset_index(drop=True)
X_main = X_base.copy()
X_main['_osm_id_txt'] = ids.values
mask_outlier = gdf['tobler_qc_excluded'].astype(bool).reset_index(drop=True)
print('QC cases found:', int(mask_outlier.sum()))
assert int(mask_outlier.sum()) == 4

def fit_pca_uai(dataframe):
    X = dataframe[PCA_VARS].astype(float).copy()
    pca = PCA(random_state=SEED).fit(X)
    coeff = pca.components_[0].copy()
    if coeff.sum() < 0:
        coeff *= -1
    w = np.abs(coeff)
    w = w / w.sum()
    raw = X.mul(pd.Series(w, index=PCA_VARS), axis=1).sum(axis=1)
    uai = MinMaxScaler().fit_transform(raw.to_numpy().reshape(-1, 1)).ravel()
    return {'pca': pca, 'coeff': pd.Series(coeff, index=PCA_VARS), 'weights': pd.Series(w, index=PCA_VARS), 'uai': uai, 'pc1': float(pca.explained_variance_ratio_[0]), 'pc12': float(pca.explained_variance_ratio_[:2].sum())}
res_full = fit_pca_uai(X_main)
X_sem4 = X_main.loc[~mask_outlier].reset_index(drop=True).copy()
res_sem4 = fit_pca_uai(X_sem4)
assert len(X_sem4) == len(X_main) - 4
weight_cmp = pd.DataFrame({'weight_full': res_full['weights'], 'weight_without_4': res_sem4['weights']})
weight_cmp['delta_weight'] = weight_cmp['weight_without_4'] - weight_cmp['weight_full']
weight_cmp['abs_delta_weight'] = weight_cmp['delta_weight'].abs()
uai_full_df = pd.DataFrame({'osm_id': X_main['_osm_id_txt'].values, 'uai_full': res_full['uai']})
uai_sem4_df = pd.DataFrame({'osm_id': X_sem4['_osm_id_txt'].values, 'uai_without_4': res_sem4['uai']})
uai_cmp = uai_sem4_df.merge(uai_full_df, on='osm_id', how='inner', validate='one_to_one')
pear_cc = pearsonr(uai_cmp['uai_full'], uai_cmp['uai_without_4']).statistic
spear_cc = spearmanr(uai_cmp['uai_full'], uai_cmp['uai_without_4']).statistic
uai_cmp['abs_delta_uai'] = (uai_cmp['uai_without_4'] - uai_cmp['uai_full']).abs()
uai_cmp['rank_full'] = uai_cmp['uai_full'].rank(method='average', ascending=True)
uai_cmp['rank_without_4'] = uai_cmp['uai_without_4'].rank(method='average', ascending=True)
uai_cmp['abs_delta_rank'] = (uai_cmp['rank_without_4'] - uai_cmp['rank_full']).abs()
set_full_cc, _, target_cc = tie_aware_ids(uai_cmp.rename(columns={'uai_full': 'value'}), 'value', id_col='osm_id', q=PRIORITY_Q)
set_sem4_cc, _, _ = tie_aware_ids(uai_cmp.rename(columns={'uai_without_4': 'value'}), 'value', id_col='osm_id', q=PRIORITY_Q)
inter_cc = set_full_cc & set_sem4_cc
ret_cc = len(inter_cc) / len(set_full_cc)
jac_cc = len(inter_cc) / len(set_full_cc | set_sem4_cc)
complete_case_summary = pd.DataFrame({'metric': ['N_full', 'N_without_4', 'PC1_full', 'PC1_without_4', 'delta_PC1', 'PC1_PC2_full', 'PC1_PC2_without_4', 'delta_PC1_PC2', 'max_abs_delta_weight', 'Pearson_UAI', 'Spearman_UAI', 'MAE_abs_delta_UAI', 'P95_abs_delta_UAI', 'P99_abs_delta_UAI', 'max_abs_delta_UAI', 'median_abs_delta_rank', 'P95_abs_delta_rank', 'P99_abs_delta_rank', 'max_abs_delta_rank', 'priority_full_tieaware_n', 'priority_without_4_tieaware_n', 'priority_retention_pct', 'priority_Jaccard'], 'value': [len(X_main), len(X_sem4), res_full['pc1'], res_sem4['pc1'], res_sem4['pc1'] - res_full['pc1'], res_full['pc12'], res_sem4['pc12'], res_sem4['pc12'] - res_full['pc12'], weight_cmp['abs_delta_weight'].max(), pear_cc, spear_cc, uai_cmp['abs_delta_uai'].mean(), uai_cmp['abs_delta_uai'].quantile(0.95), uai_cmp['abs_delta_uai'].quantile(0.99), uai_cmp['abs_delta_uai'].max(), uai_cmp['abs_delta_rank'].median(), uai_cmp['abs_delta_rank'].quantile(0.95), uai_cmp['abs_delta_rank'].quantile(0.99), uai_cmp['abs_delta_rank'].max(), len(set_full_cc), len(set_sem4_cc), ret_cc * 100, jac_cc]})
print('\nWeights: full vs without four QC cases')
display(weight_cmp.round(8))
print('\nComplete-case robustness summary')
display(complete_case_summary.round(10))
weight_cmp.to_csv(RESULTS_TABLES / '06_complete_case_weight_sensitivity.csv')
complete_case_summary.to_csv(RESULTS_TABLES / '07_complete_case_robustness.csv', index=False)

## 7. AAVI and final descriptive statistics

AAVI combines the estimated population aged 65+ with the complement of UAI: `AAVI = population_65plus × (1 − UAI)`. Population is not included in the PCA; it is introduced only after UAI construction.


In [ ]:
gdf['AAVI'] = gdf['pop_64_mais'] * (1 - gdf['uai'])
summary = pd.Series({'n_buildings': len(gdf), 'n_no_services': int(gdf['no_destinations'].sum()), 'pct_no_services': float(gdf['no_destinations'].mean() * 100), 'mean_services': float(gdf['numero_servicos_proximos'].mean()), 'median_services': float(gdf['numero_servicos_proximos'].median()), 'mean_diversity': float((gdf[SERVICE_COLS] > 0).sum(axis=1).mean()), 'median_diversity': float((gdf[SERVICE_COLS] > 0).sum(axis=1).median()), 'mean_distance_services_m': float(gdf['distancia_media_servicos'].mean()), 'mean_tobler07_min': float(gdf['tempo_medio_tobler07_qc_seg'].mean() / 60), 'median_tobler07_min': float(gdf['tempo_medio_tobler07_qc_seg'].median() / 60), 'population_65plus': float(gdf['pop_64_mais'].sum()), 'corr_UAI_population65_pearson': float(pearsonr(gdf['uai'], gdf['pop_64_mais']).statistic), 'corr_UAI_population65_p': float(pearsonr(gdf['uai'], gdf['pop_64_mais']).pvalue)})
print(summary)
coverage_rows = []
for c in SERVICE_COLS:
    n = int((gdf[c].fillna(0) > 0).sum())
    coverage_rows.append({'service': c, 'buildings_with_access': n, 'percentage': 100 * n / len(gdf)})
coverage = pd.DataFrame(coverage_rows).sort_values('percentage', ascending=False)
print('\nService coverage:')
display(coverage.round(2))
summary.rename('value').to_csv(RESULTS_TABLES / '08_descriptive_summary.csv')
coverage.to_csv(RESULTS_TABLES / '09_service_coverage.csv', index=False)

## 8. Spatial dependence: Moran, LISA and KNN connectivity

Global Moran's I is evaluated for `k = 8, 12, 16, 20, 24, 32`. For each specification, the notebook records both Moran's I and the number of connected components in the KNN graph. The main specification is `k = 20`, because it is the **smallest tested neighbourhood that produces a fully connected building-level spatial-weights graph** while leaving the substantive autocorrelation pattern virtually unchanged. The same `k = 20` specification is used for LISA and, where the number of units permits, for the subsequent multiscale Moran comparisons.

Disconnected-graph warnings generated for the deliberately tested lower-k specifications are suppressed only because connectivity is calculated and reported explicitly in the diagnostic table; they are not ignored analytically.


In [ ]:
def knn_weights(gdf_in, k):
    """Create a row-standardised KNN spatial-weights matrix."""
    if len(gdf_in) <= k:
        raise ValueError(f'Cannot build KNN weights with k={k} for only {len(gdf_in)} observations.')
    pts = gdf_in.geometry.centroid
    coords = np.column_stack([pts.x.to_numpy(), pts.y.to_numpy()])
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='The weights matrix is not fully connected')
        w = KNN.from_array(coords, k=k)
    w.transform = 'R'
    return w

def weights_components(w):
    """Return the number of connected components in a spatial-weights graph."""
    n_components, component_labels = connected_components(w.sparse, directed=False)
    return (int(n_components), component_labels)

def moran_knn(gdf_in, value_col, k=20, permutations=999, seed=42):
    """Calculate Global Moran's I using row-standardised KNN weights."""
    np.random.seed(seed)
    w = knn_weights(gdf_in, k)
    m = Moran(gdf_in[value_col].to_numpy(), w, permutations=permutations)
    return (m, w)
knn_rows = []
for k in K_SENSITIVITY:
    w = knn_weights(gdf, k)
    n_components, _ = weights_components(w)
    np.random.seed(SEED)
    m = Moran(gdf['uai'].to_numpy(), w, permutations=PERMUTATIONS)
    knn_rows.append({'k': int(k), 'connected_components': n_components, 'fully_connected': bool(n_components == 1), 'Moran_I': float(m.I), 'pseudo_p': float(m.p_sim)})
knn_sensitivity = pd.DataFrame(knn_rows)
print('KNN connectivity and Moran sensitivity:')
display(knn_sensitivity.round(6))
knn_sensitivity.to_csv(RESULTS_TABLES / '10_knn_connectivity_moran_sensitivity.csv', index=False)
connected_tested = knn_sensitivity.loc[knn_sensitivity['fully_connected'], 'k'].tolist()
if not connected_tested:
    raise ValueError('None of the tested KNN specifications produces a fully connected graph.')
smallest_connected_k = int(min(connected_tested))
print('\nSmallest tested fully connected k:', smallest_connected_k)
print('Configured K_MAIN:', K_MAIN)
if K_MAIN != smallest_connected_k:
    raise ValueError(f'K_MAIN={K_MAIN} is not the smallest tested fully connected specification (expected k={smallest_connected_k}).')
m_main, w_main = moran_knn(gdf, 'uai', k=K_MAIN, permutations=PERMUTATIONS, seed=SEED)
n_components_main, _ = weights_components(w_main)
if n_components_main != 1:
    raise ValueError(f'Main KNN graph is not fully connected: k={K_MAIN}, components={n_components_main}.')
print('\nMain KNN connectivity:')
print('k =', K_MAIN, '| connected components =', n_components_main)
print('\nMain Moran specification: k =', K_MAIN)
print('Moran I:', round(float(m_main.I), 6), '| pseudo-p:', float(m_main.p_sim))
np.random.seed(SEED)
lisa = Moran_Local(gdf['uai'].to_numpy(), w_main, permutations=PERMUTATIONS)
gdf['lisa_q'] = lisa.q
gdf['lisa_p'] = lisa.p_sim
sig = gdf['lisa_p'] <= 0.05
lisa_labels = np.full(len(gdf), 'Not significant', dtype=object)
lisa_labels[sig & (gdf['lisa_q'] == 1)] = 'High-High'
lisa_labels[sig & (gdf['lisa_q'] == 2)] = 'Low-High'
lisa_labels[sig & (gdf['lisa_q'] == 3)] = 'Low-Low'
lisa_labels[sig & (gdf['lisa_q'] == 4)] = 'High-Low'
gdf['lisa_cluster'] = lisa_labels
print('\nLISA clusters:')
print(gdf['lisa_cluster'].value_counts())
assert n_components_main == 1
assert np.isfinite(m_main.I)
assert 0 <= m_main.p_sim <= 1
assert gdf['lisa_cluster'].notna().all()
print('\nSpatial-weights validation: OK')

## 9. Spatial aggregation functions

All regular grids share a common 500 m anchor. Building centroids determine grid membership. BGRI assignment uses centroids in EPSG:3763, repairs invalid BGRI geometries if required, dissolves duplicate subsection identifiers, uses `intersects` for the primary join, and allows a small nearest-polygon fallback only for residual boundary mismatches. MAUP analysis stops if final BGRI coverage is below 99%.


Section 10 reconstructs its own `build` object from the validated `gdf`, but these aggregation helper functions and grid-origin constants must be executed first.


In [ ]:
build = gdf.to_crs(CRS_PROJECTED).copy()
minx, miny, maxx, maxy = build.total_bounds
GRID_ANCHOR = 500
GRID_ORIGIN_X = np.floor(minx / GRID_ANCHOR) * GRID_ANCHOR
GRID_ORIGIN_Y = np.floor(miny / GRID_ANCHOR) * GRID_ANCHOR
print('Common grid origin:', GRID_ORIGIN_X, GRID_ORIGIN_Y)

def aggregate_grid(buildings, size, value_col='uai'):
    b = buildings.copy().to_crs(CRS_PROJECTED)
    pts = b.geometry.centroid
    ix = np.floor((pts.x - GRID_ORIGIN_X) / size).astype(int)
    iy = np.floor((pts.y - GRID_ORIGIN_Y) / size).astype(int)
    unit_ids = [f'G{size}_{a}_{bb}' for a, bb in zip(ix, iy)]
    assignment = pd.DataFrame({'osm_id': b['osm_id'].values, 'unit_id': unit_ids, value_col: b[value_col].values, 'pop_64_mais': b['pop_64_mais'].values, 'ix': ix.values, 'iy': iy.values})
    stats = assignment.groupby(['unit_id', 'ix', 'iy'], as_index=False).agg(unit_mean=(value_col, 'mean'), unit_std=(value_col, 'std'), unit_min=(value_col, 'min'), unit_max=(value_col, 'max'), n_buildings=('osm_id', 'count'), pop_64_mais=('pop_64_mais', 'sum'))
    stats['unit_range'] = stats['unit_max'] - stats['unit_min']
    stats['geometry'] = [box(GRID_ORIGIN_X + i * size, GRID_ORIGIN_Y + j * size, GRID_ORIGIN_X + (i + 1) * size, GRID_ORIGIN_Y + (j + 1) * size) for i, j in zip(stats['ix'], stats['iy'])]
    units = gpd.GeoDataFrame(stats, geometry='geometry', crs=CRS_PROJECTED)
    assignment = assignment.merge(units[['unit_id', 'unit_mean']], on='unit_id', how='left')
    assert len(assignment) == len(b)
    assert assignment['osm_id'].nunique() == len(b)
    assert assignment['unit_mean'].notna().all()
    return (units, assignment)

def aggregate_bgri(buildings, bgri_file=BGRI_FILE, id_col=BGRI_ID_COL, value_col='uai', nearest_tolerance_m=20.0):
    b = buildings.copy().to_crs(CRS_PROJECTED)
    pts = b[['osm_id', value_col, 'pop_64_mais', 'geometry']].copy()
    pts['geometry'] = pts.geometry.centroid
    bg = gpd.read_file(bgri_file).to_crs(CRS_PROJECTED)
    if id_col not in bg.columns:
        raise KeyError(f'{id_col} is not present in the BGRI file. Columns: {bg.columns.tolist()}')
    bg = bg[bg.geometry.notna() & ~bg.geometry.is_empty].copy()
    invalid = ~bg.geometry.is_valid
    print('Invalid BGRI geometries before repair:', int(invalid.sum()))
    if invalid.any():
        bg.loc[invalid, 'geometry'] = bg.loc[invalid, 'geometry'].buffer(0)
    print('Invalid BGRI geometries after repair:', int((~bg.geometry.is_valid).sum()))
    bg = bg[[id_col, 'geometry']].dissolve(by=id_col, as_index=False)
    print('BGRI units after dissolve:', len(bg))
    joined = gpd.sjoin(pts, bg, how='left', predicate='intersects')
    joined = joined.sort_values(['osm_id', id_col]).drop_duplicates(subset='osm_id').drop(columns='index_right', errors='ignore')
    n_direct = int(joined[id_col].notna().sum())
    print('Direct BGRI assignment:', n_direct, '/', len(pts), '=', round(100 * n_direct / len(pts), 3), '%')
    unassigned_ids = joined.loc[joined[id_col].isna(), 'osm_id'].tolist()
    if unassigned_ids:
        missing = pts[pts['osm_id'].isin(unassigned_ids)].copy()
        near = gpd.sjoin_nearest(missing, bg, how='left', max_distance=nearest_tolerance_m, distance_col='_dist_bgri_m')
        near = near.sort_values(['osm_id', '_dist_bgri_m', id_col]).drop_duplicates(subset='osm_id')
        valid_near = near[near[id_col].notna()].copy()
        print('Recovered by nearest join <=', nearest_tolerance_m, 'm:', len(valid_near))
        if len(valid_near):
            print(valid_near['_dist_bgri_m'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
            mapping = valid_near.set_index('osm_id')[id_col]
            joined = joined.set_index('osm_id')
            common = joined.index.intersection(mapping.index)
            joined.loc[common, id_col] = mapping.loc[common]
            joined = joined.reset_index()
    coverage_final = float(joined[id_col].notna().mean() * 100)
    print('Final BGRI coverage:', round(coverage_final, 3), '%')
    if coverage_final < 99.0:
        raise ValueError('BGRI coverage is below 99%; MAUP analysis must not proceed.')
    assigned = joined.dropna(subset=[id_col]).copy()
    assigned['unit_id'] = assigned[id_col].astype(str)
    stats = assigned.groupby('unit_id', as_index=False).agg(unit_mean=(value_col, 'mean'), unit_std=(value_col, 'std'), unit_min=(value_col, 'min'), unit_max=(value_col, 'max'), n_buildings=('osm_id', 'count'), pop_64_mais=('pop_64_mais', 'sum'))
    stats['unit_range'] = stats['unit_max'] - stats['unit_min']
    bg_units = bg.copy()
    bg_units['unit_id'] = bg_units[id_col].astype(str)
    units = bg_units.merge(stats, on='unit_id', how='inner')
    assignment = assigned[['osm_id', 'unit_id', value_col]].merge(units[['unit_id', 'unit_mean']], on='unit_id', how='left')
    assert assignment['osm_id'].is_unique
    assert assignment['unit_mean'].notna().all()
    return (units, assignment, coverage_final)

## 10. Final MAUP analysis with tie-aware priorities

The building-level reference is the lower decile of UAI **including all ties at the cutoff**. Each aggregated representation is assigned back to buildings, and its lower decile is also made tie-aware. This avoids an arbitrary split of buildings sharing an identical unit mean.

The main variance reported for direct continuity with the manuscript is variance among unit means. The reconstructed-building variance is retained as an additional like-for-like sensitivity measure.


In [ ]:
required_runtime_objects = ['gdf', 'aggregate_grid', 'aggregate_bgri', 'moran_knn', 'weights_components', 'GRID_ORIGIN_X', 'GRID_ORIGIN_Y']
missing_runtime_objects = [name for name in required_runtime_objects if name not in globals()]
if missing_runtime_objects:
    raise RuntimeError('Section 10 requires Sections 1-9 to be executed first. Missing runtime objects: ' + ', '.join(missing_runtime_objects))
build = gdf.to_crs(CRS_PROJECTED).copy()
if 'uai' not in build.columns:
    raise KeyError("The validated building GeoDataFrame does not contain 'uai'. Run the PCA/UAI section before the multiscale analysis.")
if build['osm_id'].duplicated().any():
    raise ValueError('Duplicate osm_id values detected before MAUP analysis.')
if build['uai'].isna().any():
    raise ValueError('Missing UAI values detected before MAUP analysis.')
print('MAUP input reconstructed from gdf:', len(build), 'buildings')
var_build = float(build['uai'].var(ddof=1))
N_BUILD = len(build)
N_PRIORITY_TARGET = int(np.ceil(N_BUILD * PRIORITY_Q))
build_sorted = build[['osm_id', 'uai']].sort_values(['uai', 'osm_id']).copy()
cut_build = float(build_sorted.iloc[N_PRIORITY_TARGET - 1]['uai'])
priority_reference_ids = set(build_sorted.loc[build_sorted['uai'] <= cut_build, 'osm_id'])
N_PRIORITY_REFERENCE = len(priority_reference_ids)
print('Buildings:', N_BUILD)
print('Nominal lower-decile target:', N_PRIORITY_TARGET)
print('Tie-aware building priorities:', N_PRIORITY_REFERENCE)
print('Building-level cutoff:', cut_build)
print('Extra cases due to ties:', N_PRIORITY_REFERENCE - N_PRIORITY_TARGET)

def representation_metrics_tieaware(label, units_gdf, assignment_df, coverage_pct=100.0):
    """Calculate multiscale diagnostics using tie-aware building priorities."""
    matched = build[['osm_id', 'uai']].merge(assignment_df[['osm_id', 'unit_id', 'unit_mean']], on='osm_id', how='inner', validate='one_to_one')
    coverage_real = len(matched) / len(build) * 100
    print(f'\n{label}: {len(matched)} / {len(build)} buildings ({coverage_real:.3f}% coverage)')
    if coverage_real < 99.0:
        raise ValueError(f'{label}: building coverage is below 99%.')
    var_units = float(units_gdf['unit_mean'].var(ddof=1))
    reduction_units = (var_build - var_units) / var_build * 100
    var_reconstructed = float(matched['unit_mean'].var(ddof=1))
    reduction_reconstructed = (var_build - var_reconstructed) / var_build * 100
    pear = pearsonr(matched['uai'], matched['unit_mean']).statistic
    spear = spearmanr(matched['uai'], matched['unit_mean']).statistic
    matched_sorted = matched[['osm_id', 'unit_mean']].sort_values(['unit_mean', 'osm_id']).copy()
    cut_agg = float(matched_sorted.iloc[N_PRIORITY_TARGET - 1]['unit_mean'])
    agg_priority_ids = set(matched_sorted.loc[matched_sorted['unit_mean'] <= cut_agg, 'osm_id'])
    ref_matched = priority_reference_ids & set(matched['osm_id'])
    intersection = ref_matched & agg_priority_ids
    union = ref_matched | agg_priority_ids
    retention = len(intersection) / len(ref_matched)
    jaccard = len(intersection) / len(union)
    missed = 1 - retention
    k_used = min(K_MAIN, len(units_gdf) - 1)
    units_for_moran = units_gdf.rename(columns={'unit_mean': '_value'})
    m, w = moran_knn(units_for_moran, '_value', k=k_used, permutations=PERMUTATIONS, seed=SEED)
    n_components, _ = weights_components(w)
    print(f'{label}: Moran KNN k={k_used}, connected components={n_components}')
    if n_components != 1:
        raise ValueError(f'{label}: KNN graph is not fully connected at k={k_used} (components={n_components}).')
    areas = units_gdf.geometry.area
    return {'Spatial unit': label, 'n_units': len(units_gdf), 'coverage_buildings_pct': coverage_real, 'Variance_unit_means': var_units, 'Variance_reduction_unit_means_pct': reduction_units, 'Variance_reconstructed_buildings': var_reconstructed, 'Variance_reduction_reconstructed_pct': reduction_reconstructed, 'Global_Moran_I': float(m.I), 'pseudo_p': float(m.p_sim), 'KNN_k': k_used, 'KNN_connected_components': n_components, 'Pearson_building_vs_aggregate': pear, 'Spearman_building_vs_aggregate': spear, 'Priority_target_nominal_n': N_PRIORITY_TARGET, 'Priority_reference_tieaware_n': len(ref_matched), 'Priority_aggregate_tieaware_n': len(agg_priority_ids), 'Priority_retained_n': len(intersection), 'Priority_cutoff': cut_agg, 'Priority_retention_pct': retention * 100, 'Missed_priority_pct': missed * 100, 'Jaccard_priority': jaccard, 'mean_buildings_per_unit': units_gdf['n_buildings'].mean(), 'median_buildings_per_unit': units_gdf['n_buildings'].median(), 'cv_buildings_per_unit': units_gdf['n_buildings'].std(ddof=1) / units_gdf['n_buildings'].mean(), 'mean_unit_area_km2': areas.mean() / 1000000.0, 'median_unit_area_km2': areas.median() / 1000000.0, 'mean_within_unit_range': units_gdf['unit_range'].mean(), 'median_within_unit_range': units_gdf['unit_range'].median(), 'max_within_unit_range': units_gdf['unit_range'].max(), 'mean_within_unit_std': units_gdf['unit_std'].mean()}
m_build, w_build = moran_knn(build, 'uai', k=K_MAIN, permutations=PERMUTATIONS, seed=SEED)
n_components_build, _ = weights_components(w_build)
if n_components_build != 1:
    raise ValueError(f'Building-level KNN graph is not fully connected at k={K_MAIN} (components={n_components_build}).')
build_row = {'Spatial unit': 'Building', 'n_units': len(build), 'coverage_buildings_pct': 100.0, 'Variance_unit_means': var_build, 'Variance_reduction_unit_means_pct': 0.0, 'Variance_reconstructed_buildings': var_build, 'Variance_reduction_reconstructed_pct': 0.0, 'Global_Moran_I': float(m_build.I), 'pseudo_p': float(m_build.p_sim), 'KNN_k': K_MAIN, 'KNN_connected_components': n_components_build, 'Pearson_building_vs_aggregate': 1.0, 'Spearman_building_vs_aggregate': 1.0, 'Priority_target_nominal_n': N_PRIORITY_TARGET, 'Priority_reference_tieaware_n': N_PRIORITY_REFERENCE, 'Priority_aggregate_tieaware_n': N_PRIORITY_REFERENCE, 'Priority_retained_n': N_PRIORITY_REFERENCE, 'Priority_cutoff': cut_build, 'Priority_retention_pct': 100.0, 'Missed_priority_pct': 0.0, 'Jaccard_priority': 1.0, 'mean_buildings_per_unit': 1.0, 'median_buildings_per_unit': 1.0, 'cv_buildings_per_unit': 0.0, 'mean_unit_area_km2': np.nan, 'median_unit_area_km2': np.nan, 'mean_within_unit_range': 0.0, 'median_within_unit_range': 0.0, 'max_within_unit_range': 0.0, 'mean_within_unit_std': 0.0}
representations = {}
rows = [build_row]
grid_100, ass_100 = aggregate_grid(build, 100, 'uai')
representations['100 m grid'] = (grid_100, ass_100)
rows.append(representation_metrics_tieaware('100 m grid', grid_100, ass_100, 100.0))
bgri_units, ass_bgri, bgri_coverage = aggregate_bgri(build, BGRI_FILE, BGRI_ID_COL, 'uai')
representations['BGRI statistical subsections'] = (bgri_units, ass_bgri)
rows.append(representation_metrics_tieaware('BGRI statistical subsections', bgri_units, ass_bgri, bgri_coverage))
for size in [250, 500]:
    units_g, ass_g = aggregate_grid(build, size, 'uai')
    label = f'{size} m grid'
    representations[label] = (units_g, ass_g)
    rows.append(representation_metrics_tieaware(label, units_g, ass_g, 100.0))
multiscale = pd.DataFrame(rows)
order = ['Building', '100 m grid', 'BGRI statistical subsections', '250 m grid', '500 m grid']
multiscale['Spatial unit'] = pd.Categorical(multiscale['Spatial unit'], categories=order, ordered=True)
multiscale = multiscale.sort_values('Spatial unit').reset_index(drop=True)
main_cols = ['Spatial unit', 'n_units', 'KNN_k', 'KNN_connected_components', 'Variance_unit_means', 'Variance_reduction_unit_means_pct', 'Variance_reconstructed_buildings', 'Variance_reduction_reconstructed_pct', 'Global_Moran_I', 'Pearson_building_vs_aggregate', 'Priority_reference_tieaware_n', 'Priority_aggregate_tieaware_n', 'Priority_retention_pct', 'Missed_priority_pct', 'Jaccard_priority']
print('\n================ FINAL MULTISCALE RESULTS ================')
display(multiscale[main_cols].round(4))
print('\nKNN connectivity across spatial representations:')
connectivity_table = multiscale[['Spatial unit', 'n_units', 'KNN_k', 'KNN_connected_components']].copy()
display(connectivity_table)
assert connectivity_table['KNN_connected_components'].eq(1).all()
print('All building and aggregated KNN graphs are fully connected under the reported specification: OK')
multiscale.to_csv(RESULTS_TABLES / '11_multiscale_results_tieaware.csv', index=False)
multiscale[main_cols].to_csv(RESULTS_TABLES / '12_main_maup_table.csv', index=False)
connectivity_table.to_csv(RESULTS_TABLES / '12b_multiscale_knn_connectivity.csv', index=False)
for label, (units_gdf, assignment_df) in representations.items():
    safe = label.lower().replace(' ', '_').replace('/', '_')
    units_gdf.to_file(RESULTS_SPATIAL / f'units_{safe}.gpkg', driver='GPKG')
    assignment_df.to_csv(RESULTS_TABLES / f'assignment_{safe}.csv', index=False)

## 11. Scale, zoning and within-unit heterogeneity diagnostics

This section reports unit geometry, building counts and within-unit UAI heterogeneity for the four aggregated representations. It contains no historical comparison with earlier manuscript versions.


In [ ]:
zoning_cols = ['Spatial unit', 'n_units', 'coverage_buildings_pct', 'mean_unit_area_km2', 'median_unit_area_km2', 'mean_buildings_per_unit', 'median_buildings_per_unit', 'cv_buildings_per_unit', 'mean_within_unit_range', 'median_within_unit_range', 'max_within_unit_range', 'Variance_reduction_unit_means_pct', 'Variance_reduction_reconstructed_pct', 'Pearson_building_vs_aggregate', 'Spearman_building_vs_aggregate', 'Priority_retention_pct', 'Missed_priority_pct', 'Jaccard_priority']
zoning = multiscale.loc[multiscale['Spatial unit'].astype(str) != 'Building', zoning_cols].copy()
display(zoning.round(4))
zoning.to_csv(RESULTS_TABLES / '13_scale_zoning_heterogeneity.csv', index=False)

## 12. Sensitivity of intervention priorities to the operational cutoff

Priority-retention and Jaccard analyses are repeated at lower 5%, 10% and 20% UAI cutoffs. The same tie-aware rule is used at every threshold.


In [ ]:
def priority_set_tieaware(df_in, id_col, value_col, q):
    tmp = df_in[[id_col, value_col]].sort_values([value_col, id_col]).copy()
    n_target = int(np.ceil(len(tmp) * q))
    cutoff = float(tmp.iloc[n_target - 1][value_col])
    ids = set(tmp.loc[tmp[value_col] <= cutoff, id_col])
    return (ids, cutoff, n_target)
threshold_rows = []
for q in PRIORITY_THRESHOLDS:
    reference_ids_q, reference_cutoff_q, target_q = priority_set_tieaware(build, 'osm_id', 'uai', q)
    threshold_rows.append({'priority_threshold_pct': q * 100, 'Spatial unit': 'Building', 'nominal_target_n': target_q, 'reference_tieaware_n': len(reference_ids_q), 'aggregate_tieaware_n': len(reference_ids_q), 'priority_retention_pct': 100.0, 'missed_priority_pct': 0.0, 'Jaccard_priority': 1.0})
    for label, (_, assignment_df) in representations.items():
        matched = build[['osm_id', 'uai']].merge(assignment_df[['osm_id', 'unit_mean']], on='osm_id', how='inner', validate='one_to_one')
        aggregate_ids_q, _, _ = priority_set_tieaware(matched, 'osm_id', 'unit_mean', q)
        ref_matched_q = reference_ids_q & set(matched['osm_id'])
        intersection_q = ref_matched_q & aggregate_ids_q
        union_q = ref_matched_q | aggregate_ids_q
        retention_q = len(intersection_q) / len(ref_matched_q)
        jaccard_q = len(intersection_q) / len(union_q)
        threshold_rows.append({'priority_threshold_pct': q * 100, 'Spatial unit': label, 'nominal_target_n': target_q, 'reference_tieaware_n': len(ref_matched_q), 'aggregate_tieaware_n': len(aggregate_ids_q), 'priority_retention_pct': retention_q * 100, 'missed_priority_pct': (1 - retention_q) * 100, 'Jaccard_priority': jaccard_q})
priority_threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_order = ['Building', '100 m grid', 'BGRI statistical subsections', '250 m grid', '500 m grid']
priority_threshold_sensitivity['Spatial unit'] = pd.Categorical(priority_threshold_sensitivity['Spatial unit'], categories=threshold_order, ordered=True)
priority_threshold_sensitivity = priority_threshold_sensitivity.sort_values(['priority_threshold_pct', 'Spatial unit']).reset_index(drop=True)
display(priority_threshold_sensitivity.round(4))
priority_threshold_sensitivity.to_csv(RESULTS_TABLES / '14_priority_threshold_sensitivity.csv', index=False)
for q in PRIORITY_THRESHOLDS:
    sub = priority_threshold_sensitivity[priority_threshold_sensitivity['priority_threshold_pct'].eq(q * 100)].set_index('Spatial unit')
    assert sub.loc['100 m grid', 'priority_retention_pct'] >= sub.loc['500 m grid', 'priority_retention_pct']
print('Priority-threshold sensitivity validation: OK')

## 13. Reproducible manuscript figures

The figures use only local data and do not depend on online basemaps. Figure 8 is selected reproducibly: the 250 m grid cell with the largest within-cell UAI range is used to illustrate information loss under aggregation. All three panels share the same UAI colour scale.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
gdf.plot(column='uai', ax=ax, cmap='viridis', linewidth=0, legend=True)
ax.set_title('Building-level Urban Attractiveness Index')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'Figure_4_UAI_building_level.png', dpi=600, bbox_inches='tight')
plt.show()
lisa_categories = ['High-High', 'Low-Low', 'Low-High', 'High-Low', 'Not significant']
fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
for label in lisa_categories:
    sub = gdf[gdf['lisa_cluster'].eq(label)]
    if len(sub):
        sub.plot(ax=ax, linewidth=0, markersize=1, label=label)
ax.set_title(f'Local spatial association of building-level UAI (KNN k={K_MAIN})')
ax.set_axis_off()
ax.legend(frameon=False, markerscale=5)
plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'Figure_6_LISA_UAI_building_level.png', dpi=600, bbox_inches='tight')
plt.show()
grid100_units, _ = representations['100 m grid']
bgri_units_plot, _ = representations['BGRI statistical subsections']
grid250_units, _ = representations['250 m grid']
grid500_units, _ = representations['500 m grid']
vmin = float(build['uai'].min())
vmax = float(build['uai'].max())
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
plot_specs = [(grid100_units, '100 m grid'), (bgri_units_plot, 'BGRI statistical subsections'), (grid250_units, '250 m grid'), (grid500_units, '500 m grid')]
for ax, (units, title) in zip(axes.ravel(), plot_specs):
    units.plot(column='unit_mean', ax=ax, cmap='viridis', vmin=vmin, vmax=vmax, linewidth=0)
    ax.set_title(title)
    ax.set_axis_off()
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
cbar.set_label('Urban Attractiveness Index')
plt.savefig(RESULTS_FIGURES / 'Figure_7_UAI_aggregated_representations.png', dpi=600, bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
gdf.plot(column='AAVI', ax=ax, cmap='magma', linewidth=0, legend=True)
ax.set_title('Building-level Ageing Accessibility Vulnerability Index')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'Figure_9_AAVI_building_level.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
bgri_units_fig, _ = representations['BGRI statistical subsections']
grid250_units, _ = representations['250 m grid']
example = grid250_units.sort_values('unit_range', ascending=False).iloc[0]
example_geometry = example.geometry
minx, miny, maxx, maxy = example_geometry.bounds
MARGIN = 60.0
xmin, xmax = (minx - MARGIN, maxx + MARGIN)
ymin, ymax = (miny - MARGIN, maxy + MARGIN)
extent = example_geometry.buffer(MARGIN)
build_local = build[build.geometry.intersects(extent)].copy()
bgri_local = bgri_units_fig[bgri_units_fig.geometry.intersects(extent)].copy()
grid250_local = grid250_units[grid250_units.geometry.intersects(extent)].copy()
vmin, vmax = (float(build['uai'].min()), float(build['uai'].max()))
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
build_local.plot(column='uai', ax=axes[0], cmap='viridis', vmin=vmin, vmax=vmax, linewidth=0, legend=False)
axes[0].set_title('A. Building level')
bgri_local.plot(column='unit_mean', ax=axes[1], cmap='viridis', vmin=vmin, vmax=vmax, edgecolor='black', linewidth=0.4, legend=False)
axes[1].set_title('B. BGRI statistical subsections')
grid250_local.plot(column='unit_mean', ax=axes[2], cmap='viridis', vmin=vmin, vmax=vmax, edgecolor='black', linewidth=0.4, legend=False)
gpd.GeoSeries([example_geometry], crs=CRS_PROJECTED).boundary.plot(ax=axes[2], linewidth=2)
axes[2].set_title('C. 250 m regular grid')
for ax in axes:
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal')
    ax.set_axis_off()
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label('Urban Attractiveness Index')
plt.savefig(RESULTS_FIGURES / 'Figure_8_building_BGRI_grid250.png', dpi=600, bbox_inches='tight')
plt.show()
print('Selected 250 m cell:', example['unit_id'])
print('Cell mean UAI:', round(float(example['unit_mean']), 6))
print('Within-cell UAI range:', round(float(example['unit_range']), 6))
print('Buildings in selected cell:', int(example['n_buildings']))

## 13. Export the analytical master dataset and run final assertions

The final master dataset contains the validated UAI, AAVI, QC variables and geometry. Assertions are intentionally strict: if any one fails, the exported results should not be used in the manuscript until the discrepancy is resolved.


## 14. Manuscript-alignment checks

These checks are intentionally strict. They verify that the notebook reproduces the analytical universe and the principal numerical results reported in the submitted manuscript. Small floating-point tolerances are allowed where appropriate.


In [ ]:
if STRICT_MANUSCRIPT_VALIDATION:
    assert len(gdf) == 31664
    assert int(gdf['no_destinations'].sum()) == 420
    assert int(gdf['pop_64_mais'].sum()) == 58203
    assert np.isclose(float(kmo_total), 0.773, atol=0.001)
    assert np.isclose(float(explained[0]), 0.4975, atol=0.0006)
    assert np.isclose(float(explained[:2].sum()), 0.6762, atol=0.0006)
    assert np.isclose(float(pear_eq), 0.973, atol=0.001)
    assert np.isclose(float(spear_eq), 0.987, atol=0.001)
    assert np.isclose(float(priority_ret_equal * 100), 84.79, atol=0.05)
    assert np.isclose(float(priority_jaccard_equal), 0.736, atol=0.002)
    expected_multiscale = {'Building': {'Global_Moran_I': 0.979}, '100 m grid': {'Variance_unit_means': 0.0331, 'Global_Moran_I': 0.917, 'Priority_retention_pct': 88.61, 'Jaccard_priority': 0.795}, 'BGRI statistical subsections': {'Variance_unit_means': 0.0307, 'Global_Moran_I': 0.684, 'Priority_retention_pct': 57.97, 'Missed_priority_pct': 42.03, 'Jaccard_priority': 0.406}, '250 m grid': {'Variance_unit_means': 0.0296, 'Global_Moran_I': 0.776, 'Priority_retention_pct': 77.75, 'Jaccard_priority': 0.625}, '500 m grid': {'Variance_unit_means': 0.0264, 'Global_Moran_I': 0.569, 'Priority_retention_pct': 64.53, 'Jaccard_priority': 0.476}}
    ms = multiscale.copy()
    ms['Spatial unit'] = ms['Spatial unit'].astype(str)
    ms = ms.set_index('Spatial unit')
    for label, checks in expected_multiscale.items():
        for column, expected in checks.items():
            actual = float(ms.loc[label, column])
            tolerance = 0.002 if abs(expected) < 2 else 0.06
            assert np.isclose(actual, expected, atol=tolerance), f'{label} / {column}: expected {expected}, found {actual}'
    print('Manuscript-alignment checks: OK')
else:
    print('STRICT_MANUSCRIPT_VALIDATION is disabled.')

In [ ]:
gdf['pca_proximity'] = X_norm['proximity'].values
master_out = gdf.copy()
master_out['geometry_wkt'] = master_out.geometry.to_wkt()
master_csv = PROCESSED_DATA / 'porto_building_uai_aavi_final.csv'
master_out.drop(columns='geometry').to_csv(master_csv, index=False)
gdf.to_file(PROCESSED_DATA / 'porto_building_uai_aavi_final.gpkg', driver='GPKG')
print('=' * 80)
print('FINAL VALIDATION')
print('=' * 80)
print('Buildings:', len(gdf))
print('Unique osm_id:', gdf['osm_id'].nunique())
print('No-service buildings:', int(gdf['no_destinations'].sum()))
print('Zero-minimum snapping flags:', int(gdf['minimo_zero_snap'].sum()))
print('Invalid Tobler QC flags:', int(gdf['tobler_qc_excluded'].sum()))
print('PCA NaN:', int(X_norm.isna().sum().sum()))
print('UAI NaN:', int(gdf['uai'].isna().sum()))
print('UAI range:', float(gdf['uai'].min()), float(gdf['uai'].max()))
print('Negative AAVI:', int((gdf['AAVI'] < 0).sum()))
print('BGRI coverage:', round(float(bgri_coverage), 3), '%')
assert len(gdf) == gdf['osm_id'].nunique()
assert X_norm.isna().sum().sum() == 0
assert gdf['uai'].notna().all()
assert np.isclose(gdf['uai'].min(), 0.0)
assert np.isclose(gdf['uai'].max(), 1.0)
assert (gdf['AAVI'] >= 0).all()
assert bgri_coverage >= 99.0
assert N_PRIORITY_REFERENCE >= N_PRIORITY_TARGET
assert multiscale['coverage_buildings_pct'].ge(99.0).all()
assert multiscale['Priority_retention_pct'].between(0, 100).all()
assert multiscale['Missed_priority_pct'].between(0, 100).all()
assert multiscale['Jaccard_priority'].between(0, 1).all()
assert np.allclose(multiscale['Priority_retention_pct'] + multiscale['Missed_priority_pct'], 100.0, atol=1e-08)
assert np.isclose(res_full['weights'].sum(), 1.0)
assert np.isclose(res_sem4['weights'].sum(), 1.0)
assert 0 <= ret_cc <= 1 and 0 <= jac_cc <= 1
print('\nAll structural checks passed.')
print('Master CSV:', master_csv.resolve())